# 01 — Análise inicial dos dados
Visão **estrutural** do dataset completo (tipos, valores únicos, ausentes, duplicadas, balanceamento do alvo).
Aqui **não se tomam decisões estatísticas** (médias, correlações, outliers, cestas): isso é feito na EDA (`03`), usando **somente o conjunto de treino**.

O CSV em `data/raw/` **nunca é alterado**.

In [ ]:
# Localiza a raiz do repositório (funciona no VS Code e no Colab) e a coloca no sys.path
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src import config as cfg
from src.data_loader import load_raw

cfg.ensure_dirs()
df = load_raw()
print("Nº de instâncias:", len(df), "| colunas:", df.shape[1])
df.head()

## Tipos, valores únicos e identificadores

In [ ]:
info = pd.DataFrame({"dtype": df.dtypes.astype(str), "nunique": df.nunique(), "ausentes": df.isna().sum()})
info.to_csv(cfg.REPORTS_TABLES / "01_resumo_colunas.csv")
info

O dataset **não tem coluna identificadora** (CPF, nome, customerID), então nenhuma coluna é removida. Para rastrear cada registro depois do particionamento/oversampling, usamos o **número da linha no CSV original** como `id_origem`.

## Valores de cada variável categórica

In [ ]:
for col in cfg.CATEGORICAL_COLS:
    print(df[col].value_counts().to_dict(), "<-", col)

Observação: `MultipleLines` tem o valor "No phone service" e as colunas de serviços de internet têm "No internet service". São **redundantes** com `PhoneService`/`InternetService`; vamos tratá-los no pré-processamento.

## Dados ausentes

In [ ]:
ausentes = df[df.isna().any(axis=1)]
print("Linhas com ausentes:", len(ausentes), "| colunas:", df.columns[df.isna().any()].tolist())
ausentes[["tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

Todas as linhas com `TotalCharges` ausente têm `tenure = 0` (cliente que ainda não completou o primeiro mês, logo nada foi cobrado). Isso sugere imputar `0` em vez de mediana/KNN. **Decisão registrada para o pré-processamento** (a regra é determinística e não depende de estatísticas do teste).

In [ ]:
print("tenure das linhas com TotalCharges ausente:", ausentes["tenure"].unique())
print("Linhas com tenure == 0 no dataset:", (df["tenure"] == 0).sum())

## Linhas duplicadas

In [ ]:
dups = df.duplicated(keep=False)
print("Linhas idênticas (todas as colunas, incluindo o alvo):", int(df.duplicated().sum()), "extras |", int(dups.sum()), "no total")

Sem identificador de cliente, linhas idênticas podem ser clientes diferentes com o mesmo perfil. **Não removemos**; o notebook `02` reporta quantas caem em conjuntos diferentes.

## Balanceamento do alvo

In [ ]:
vc = df[cfg.TARGET].value_counts()
print(vc.to_dict(), "| % churn:", round(100 * vc[cfg.POSITIVE_CLASS] / len(df), 2))
fig, ax = plt.subplots(figsize=(4, 3))
vc.plot.bar(ax=ax, color=["tab:blue", "tab:red"], rot=0)
ax.set(title="Distribuição do alvo (dataset completo)", ylabel="clientes")
for i, v in enumerate(vc):
    ax.text(i, v, str(v), ha="center", va="bottom")
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "01_balanceamento_alvo.png", dpi=150); plt.show()

**Conclusão:** base desbalanceada (~27% churn), por isso o particionamento é **estratificado** e usa **oversampling apenas em treino e validação**. Próximo: `02_particionamento_dados.ipynb`.